# Imports

In [152]:
import pandas as pd 
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE, RandomOverSampler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import numpy as np

# Loading data and exploring it 

In [153]:
train_df =pd.read_csv("D:/UNEEQ Interns/Customer chrun/customer_churn_dataset-training-master.csv")
test_df = pd.read_csv("D:/UNEEQ Interns/Customer chrun/customer_churn_dataset-testing-master.csv")

# Handle Churn column properly (fill NaN and convert to int)
train_df['Churn'] = train_df['Churn'].fillna(0).astype(int)
test_df['Churn'] = test_df['Churn'].astype(int)

In [154]:
# Training data
print("First 5 rows of the training data:")
train_df.head()

First 5 rows of the training data:


,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,2.0,30.0,Female,39.0,14.0,5.0,18.0,Standard,Annual,932.0,17.0,1
1,3.0,65.0,Female,49.0,1.0,10.0,8.0,Basic,Monthly,557.0,6.0,1
2,4.0,55.0,Female,14.0,4.0,6.0,18.0,Basic,Quarterly,185.0,3.0,1
3,5.0,58.0,Male,38.0,21.0,7.0,7.0,Standard,Monthly,396.0,29.0,1
4,6.0,23.0,Male,32.0,20.0,5.0,8.0,Basic,Monthly,617.0,20.0,1


In [155]:
train_df.describe()

,CustomerID,Age,Tenure,Usage Frequency,Support Calls,Payment Delay,Total Spend,Last Interaction,Churn
count,440832.000000,440832.000000,440832.000000,440832.000000,440832.000000,440832.000000,440832.000000,440832.000000,440833.000000
mean,225398.667955,39.373153,31.256336,15.807494,3.604437,12.965722,631.616223,14.480868,0.567106
std,129531.918550,12.442369,17.255727,8.586242,3.070218,8.258063,240.803001,8.596208,0.495477
min,2.000000,18.000000,1.000000,1.000000,0.000000,0.000000,100.000000,1.000000,0.000000
25%,113621.750000,29.000000,16.000000,9.000000,1.000000,6.000000,480.000000,7.000000,0.000000
50%,226125.500000,39.000000,32.000000,16.000000,3.000000,12.000000,661.000000,14.000000,1.000000
75%,337739.250000,48.000000,46.000000,23.000000,6.000000,19.000000,830.000000,22.000000,1.000000
max,449999.000000,65.000000,60.000000,30.000000,10.000000,30.000000,1000.000000,30.000000,1.000000


In [156]:
train_df['Churn'].value_counts()
train_df.isnull().sum()
train_df.info



<bound method DataFrame.info of         CustomerID   Age  Gender  Tenure  Usage Frequency  Support Calls  \
0              2.0  30.0  Female    39.0             14.0            5.0   
1              3.0  65.0  Female    49.0              1.0           10.0   
2              4.0  55.0  Female    14.0              4.0            6.0   
3              5.0  58.0    Male    38.0             21.0            7.0   
4              6.0  23.0    Male    32.0             20.0            5.0   
...            ...   ...     ...     ...              ...            ...   
440828    449995.0  42.0    Male    54.0             15.0            1.0   
440829    449996.0  25.0  Female     8.0             13.0            1.0   
440830    449997.0  26.0    Male    35.0             27.0            1.0   
440831    449998.0  28.0    Male    55.0             14.0            2.0   
440832    449999.0  31.0    Male    48.0             20.0            1.0   

        Payment Delay Subscription Type Contract Length

In [157]:
train_df.isnull().sum()
train_df.dtypes

CustomerID           float64
Age                  float64
Gender                object
Tenure               float64
Usage Frequency      float64
Support Calls        float64
Payment Delay        float64
Subscription Type     object
Contract Length       object
Total Spend          float64
Last Interaction     float64
Churn                  int64
dtype: object

In [158]:
# Check unique values in 'Churn' before any changes
print("Unique values in 'Churn' before processing:")
print(train_df['Churn'].unique())
print("Value counts:")
print(train_df['Churn'].value_counts())

Unique values in 'Churn' before processing:
[1 0]
Value counts:
Churn
1    249999
0    190834
Name: count, dtype: int64


# Data and feature Engineering 

In [159]:
# Handling missing values for numerical columns by filling them with the mean of each column
train_df.fillna(train_df.median(numeric_only= True), inplace=True)
test_df.fillna(train_df.median(numeric_only= True), inplace=True)

#filling missing values for categorical columns with "Na"
train_df.fillna("Na", inplace=True)
test_df.fillna("Na", inplace=True)


In [160]:
# Identify categorical columns (exclude Churn)
categorical_cols = train_df.select_dtypes(include=['object']).columns
if 'Churn' in categorical_cols:
    categorical_cols = categorical_cols.drop('Churn')

# Combine train and test for fitting encoder to ensure all categories are known
combined_df = pd.concat([train_df[categorical_cols], test_df[categorical_cols]], axis=0)

# Fit OneHotEncoder on combined data
ohe = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
ohe.fit(combined_df)

# Transform train data
encoded_train = ohe.transform(train_df[categorical_cols])
encoded_train_df = pd.DataFrame(encoded_train, columns=ohe.get_feature_names_out(categorical_cols), index=train_df.index)
train_df = train_df.drop(categorical_cols, axis=1)
train_df = pd.concat([train_df, encoded_train_df], axis=1)

# Transform test data
encoded_test = ohe.transform(test_df[categorical_cols])
encoded_test_df = pd.DataFrame(encoded_test, columns=ohe.get_feature_names_out(categorical_cols), index=test_df.index)
test_df = test_df.drop(categorical_cols, axis=1)
test_df = pd.concat([test_df, encoded_test_df], axis=1)

In [161]:
#Splitting features and target variable (drop CustomerID as it's not predictive)
x = train_df.drop(['Churn', 'CustomerID'], axis=1)
y = train_df['Churn']

# Check class distribution
print("Class distribution in y:")
print(y.value_counts())

# Use SMOTE for oversampling to balance classes
smote = SMOTE(random_state=42)
x_res, y_res = smote.fit_resample(x, y)

Class distribution in y:
Churn
1    249999
0    190834
Name: count, dtype: int64


# Modeling 

In [162]:
x_train , x_val , y_train , y_val = train_test_split(x_res, y_res, test_size=0.2, random_state=42)

# Scale the features
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_val_scaled = scaler.transform(x_val)

# Train the XGBoost model
model = XGBClassifier(scale_pos_weight=len(y_res[y_res==0])/len(y_res[y_res==1]), random_state=42, n_estimators=100)
model.fit(x_train_scaled, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [163]:
# Predict on the validation set
y_pred = model.predict(x_val_scaled)

# Evaluate the model
print(classification_report(y_val, y_pred))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00     49962
           1       1.00      1.00      1.00     50038

    accuracy                           1.00    100000
   macro avg       1.00      1.00      1.00    100000
weighted avg       1.00      1.00      1.00    100000



In [164]:
# Get top 10 important features
feature_importances = pd.Series(model.feature_importances_, index=x.columns)
top_features = feature_importances.nlargest(20)
print("Top 10 Feature Importances:")
print(top_features)

# Select only top features
selected_features = top_features.index.tolist()
print("Selected features:", selected_features)

Top 10 Feature Importances:
Contract Length_Monthly       0.291770
Support Calls                 0.230598
Total Spend                   0.214377
Payment Delay                 0.110742
Gender_Male                   0.075709
Age                           0.040362
Subscription Type_Standard    0.013963
Last Interaction              0.012715
Tenure                        0.005787
Usage Frequency               0.002629
Subscription Type_Premium     0.001266
Contract Length_Quarterly     0.000081
Gender_Na                     0.000000
Subscription Type_Na          0.000000
Contract Length_Na            0.000000
dtype: float32
Selected features: ['Contract Length_Monthly', 'Support Calls', 'Total Spend', 'Payment Delay', 'Gender_Male', 'Age', 'Subscription Type_Standard', 'Last Interaction', 'Tenure', 'Usage Frequency', 'Subscription Type_Premium', 'Contract Length_Quarterly', 'Gender_Na', 'Subscription Type_Na', 'Contract Length_Na']


In [165]:
# Redefine x with selected features
x = train_df[selected_features]
y = train_df['Churn']

# Check class distribution
print("Class distribution in y:")
print(y.value_counts())

# Use SMOTE for oversampling to balance classes
smote = SMOTE(random_state=42)
x_res, y_res = smote.fit_resample(x, y)

Class distribution in y:
Churn
1    249999
0    190834
Name: count, dtype: int64


In [166]:
x_train , x_val , y_train , y_val = train_test_split(x_res, y_res, test_size=0.2, random_state=42)

# Scale the features
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_val_scaled = scaler.transform(x_val)

# Train the XGBoost model
model = XGBClassifier(scale_pos_weight=len(y_res[y_res==0])/len(y_res[y_res==1]), random_state=42, n_estimators=100)
model.fit(x_train_scaled, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [167]:
# Prepare test data (drop target column and any prediction columns)
columns_to_drop = ['Churn', 'CustomerID']
if 'Predicted_Churn' in test_df.columns:
    columns_to_drop.append('Predicted_Churn')
x_test = test_df.drop(columns_to_drop, axis=1)
x_test = x_test[selected_features]  # Select only the important features
x_test_scaled = scaler.transform(x_test)

# Predict on the test set with probability scores
test_probabilities = model.predict_proba(x_test_scaled)
# Use threshold 0.5
threshold = 0.5
test_predictions = (test_probabilities[:, 1] >= threshold).astype(int)

# Output the feature labels
print("Features used for prediction:")
print(list(x.columns))

# Add predictions as a new column to test_df
test_df['Predicted_Churn'] = test_predictions

# Compare predictions with actual values
print("\nFirst 10 rows comparison:")
print(test_df[['Churn', 'Predicted_Churn']].head(10))

# Confusion Matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(test_df['Churn'], test_df['Predicted_Churn'])
print("\nConfusion Matrix:")
print("[[TN, FP]")
print(" [FN, TP]]")
print(cm)

# Detailed comparison for first 20 rows
print("\nDetailed comparison (first 20 rows):")
comparison_df = test_df[['Churn', 'Predicted_Churn']].copy()
comparison_df['Correct'] = comparison_df['Churn'] == comparison_df['Predicted_Churn']
print(comparison_df.head(20))

# Show first 20 rows of test_df with all columns
print("\nFirst 20 rows of test_df with predictions:")
print(test_df.head(20))

# Calculate accuracy
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(test_df['Churn'], test_df['Predicted_Churn'])
print(f"\nAccuracy on test set: {accuracy:.4f}")

# Additional metrics
from sklearn.metrics import precision_score, recall_score, f1_score
precision = precision_score(test_df['Churn'], test_df['Predicted_Churn'])
recall = recall_score(test_df['Churn'], test_df['Predicted_Churn'])
f1 = f1_score(test_df['Churn'], test_df['Predicted_Churn'])
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

# Output the predictions
print("\nTest Predictions:")
print(test_predictions)

# Summary of predictions
import pandas as pd
print("\nPrediction Summary:")
pred_counts = pd.Series(test_predictions).value_counts()
print(pred_counts)

# Show that it's not all 1s
total_predictions = len(test_predictions)
ones_count = pred_counts.get(1, 0)
zeros_count = pred_counts.get(0, 0)
print(f"\nOut of {total_predictions} predictions:")
print(f"- {ones_count} predicted as 1 (churn)")
print(f"- {zeros_count} predicted as 0 (no churn)")
print(f"- Percentage of 1s: {ones_count/total_predictions*100:.1f}%")
print(f"- Percentage of 0s: {zeros_count/total_predictions*100:.1f}%")

# Feature coefficients
print("\nFeature Coefficients:")
for feature, coef in zip(x.columns, model.feature_importances_):
    print(f"{feature}: {coef:.4f}")

Features used for prediction:
['Contract Length_Monthly', 'Support Calls', 'Total Spend', 'Payment Delay', 'Gender_Male', 'Age', 'Subscription Type_Standard', 'Last Interaction', 'Tenure', 'Usage Frequency', 'Subscription Type_Premium', 'Contract Length_Quarterly', 'Gender_Na', 'Subscription Type_Na', 'Contract Length_Na']

First 10 rows comparison:
   Churn  Predicted_Churn
0      1                1
1      0                1
2      0                1
3      0                1
4      0                1
5      0                1
6      1                1
7      0                1
8      0                1
9      0                1

Confusion Matrix:
[[TN, FP]
 [FN, TP]]
[[ 1952 31929]
 [   40 30453]]

Detailed comparison (first 20 rows):
    Churn  Predicted_Churn  Correct
0       1                1     True
1       0                1    False
2       0                1    False
3       0                1    False
4       0                1    False
5       0                1    False
6

In [168]:
# Check test data Churn distribution
print("Test data Churn distribution:")
print(test_df['Churn'].value_counts())

Test data Churn distribution:
Churn
0    33881
1    30493
Name: count, dtype: int64
